# Linear AutoEncoder and PCA equivalence

A linear under-complete auto-encoder (feedforward network with 1 hidden layer and 1 output layer with equal number of input/output), is equivalent to PCA of k components, as in it covers the same subspace, provided
- Loss function is mean squared error
- Data is mean centered

In [1]:
import torch
import torch.nn as nn 
from torchvision.datasets import MNIST
from ipywidgets import IntProgress, Label
from IPython.display import display

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

## Dataset preperation

Using MNIST dataset.

In [3]:
dataset = MNIST(root='../data/', download=True)
train, test = torch.utils.data.random_split(dataset, [50000, 10000])


trainX = train.dataset.data.reshape(-1, 28*28).type(torch.float32)/255.0
meanX = torch.mean(trainX, dim=0)
sdX  = torch.std(trainX, dim=0)

trainX = (trainX - meanX).to(device)

testX = (test.dataset.data.reshape(-1, 28*28).type(torch.float32)/255.0)
testX = (testX - meanX).to(device)

components = 32
criterion = nn.MSELoss()

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.85MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 121kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.19MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]


## Principal Component Analysis

The train data and test data  (using train's mean) have already been mean centered. It's now a simple job of finding eigenvectors of $X^TX$. Taking the largest 32 components as the principal components. The reconstruction error comes to around 0.0172.

In [4]:
def pca(X, num_components):
    XTX = torch.matmul(X.T, X) / (X.shape[0] - 1)
    
    eigenvalues, eigenvectors = torch.linalg.eigh(XTX)
    sorted_indices = torch.argsort(eigenvalues, descending=True)
    eigenvectors = eigenvectors[:, sorted_indices]

    return eigenvectors[:, :num_components]

P = pca(trainX, num_components=components)

testX_recon = testX @ P @ P.T

diff = testX_recon - testX

baseline_error = criterion(testX_recon, testX)

print(f'Baseline (PCA) Error: {baseline_error.item():.4f}')

Baseline (PCA) Error: 0.0172


## Simple Linear AutoEncoder

A linear auto encoder has been trained with 32 hidden units. The reconstruction error on test data came out to be 0.0172. Comparing the reconstructed test data using PCA and Autoencoder using mean squared loss gives approximately 0 error, which is makes hypothesis that it spans the same subspace true.

In [5]:
autoencoder = nn.Sequential(
    nn.Linear(28*28, components),
    nn.Linear(components, 28*28),
).to(device)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.05)
epochs = 1000

progress = IntProgress(min=0, max=epochs)
label = Label()

display(progress, label)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = autoencoder(trainX)
    loss = criterion(outputs, trainX)
    loss.backward()
    optimizer.step()

    progress.value += 1
    label.value = f'Epoch: {epoch + 1}/{epochs} Loss: {loss.item():.4f}'

autoencoder_vs_pca = criterion(autoencoder.forward(testX), testX_recon)

print(f'Autoencoder vs PCA Error: {autoencoder_vs_pca.item():.4f}')

IntProgress(value=0, max=1000)

Label(value='')

Autoencoder vs PCA Error: 0.0000


## Tying

Sometimes the auto encoder doesn't generalize enough (this case it does), so it's natural to do some regularization and thereby introducing the bias. Since the linear autoencoder is equivalent to PCA, one way to introduce a bias is to force the weights to be orthonormal (not necessarily eigen), by making the encoder and decoder weights transpose of each other. This halves the number of parameters required for the model.

This also gives the same reconstruction error and the reconstruction difference with PCA is negligible here too.

In [6]:
class TiedAutoEncoder(nn.Module):
    def __init__(self, input, hidden):
        super(TiedAutoEncoder, self).__init__()
        self.parameters = torch.randn(input, hidden)

    def forward(self, x):
        encoded = x @ self.parameters
        decoded = encoded @ self.parameters.T
        return decoded
    
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.05)
epochs = 1000

progress = IntProgress(min=0, max=epochs)
label = Label()

display(progress, label)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = autoencoder(trainX)
    loss = criterion(outputs, trainX)
    loss.backward()
    optimizer.step()

    progress.value += 1
    label.value = f'Epoch: {epoch + 1}/{epochs} Loss: {loss.item():.4f}'

autoencoder_vs_pca = criterion(autoencoder.forward(testX), testX_recon)

print(f'Autoencoder vs PCA Error: {autoencoder_vs_pca.item():.4f}')

IntProgress(value=0, max=1000)

Label(value='')

Autoencoder vs PCA Error: 0.0001
